In [ ]:
from groq import Groq

client = Groq(api_key="YOUR_GROQ_API_KEY")


In [9]:


models = client.models.list()
for m in models.data:
    print(m.id)


canopylabs/orpheus-v1-english
openai/gpt-oss-safeguard-20b
meta-llama/llama-prompt-guard-2-22m
allam-2-7b
whisper-large-v3-turbo
meta-llama/llama-prompt-guard-2-86m
groq/compound-mini
meta-llama/llama-4-scout-17b-16e-instruct
groq/compound
llama-3.1-8b-instant
canopylabs/orpheus-arabic-saudi
openai/gpt-oss-120b
openai/gpt-oss-20b
whisper-large-v3
llama-3.3-70b-versatile
qwen/qwen3-32b


In [3]:
INTENTS = ["greeting", "goodbye", "gratitude", "asking_mental_health_question", "out_of_scope"]


In [4]:
def build_intent_prompt(user_query: str, mode: str = "zero-shot") -> str:
    if mode == "zero-shot":
        return f"""
        Classify the intent of this user query into one of:
        {INTENTS}.
        Query: "{user_query}"
        """
    elif mode == "few-shot":
        examples = """
        Examples:
        "Hello there!" → greeting
        "Bye for now" → goodbye
        "Thanks so much" → gratitude
        "I feel anxious lately" → asking_mental_health_question
        "What’s the weather?" → out_of_scope
        """
        return f"""{examples}

        Classify: "{user_query}"
        """


In [ ]:
def classify_intent(user_query: str, mode: str = "zero-shot") -> str:
    prompt = build_intent_prompt(user_query, mode)

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",  
        messages=[
            {"role": "system", "content": "You are an intent classifier."},
            {"role": "user", "content": prompt}
        ]
    )

    # Normalize output to one of the intents
    result = response.choices[0].message.content.strip().lower()
    for intent in INTENTS:
        if intent in result:
            return intent
    return "out_of_scope"  # fallback


In [11]:
print(classify_intent("Hello there!", mode="few-shot"))  
# Expected → "greeting"

print(classify_intent("I feel anxious lately", mode="few-shot"))  
# Expected → "asking_mental_health_question"

print(classify_intent("Thanks a lot!", mode="zero-shot"))  
# Expected → "gratitude"


greeting
asking_mental_health_question
gratitude
